# 02 Preprocessing: add synthetic prices

ABO listings have no price field. This notebook adds **simulated (synthetic) prices** so price filtering (e.g. 'under $50') can be demonstrated. They are not real prices.

Prerequisites: `01_data_exploration.ipynb` saved `listings_en_raw.parquet` to Drive, and `src/preprocess/synthetic_prices.py` is committed to the repo.

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/shoptalk'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        url = f'https://{token}@github.com/anbuaru/shoptalk.git'
    except Exception:
        url = 'https://github.com/anbuaru/shoptalk.git'
    !git clone {url} {REPO}
else:
    !git -C {REPO} pull

%cd /content/shoptalk
!pip install -q -r requirements.txt

Mounted at /content/drive
Cloning into '/content/shoptalk'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 67 (delta 26), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 35.27 KiB | 8.82 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/shoptalk
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [3]:
import sys
import os
import pandas as pd

sys.path.append(REPO)
assert os.path.exists(f'{REPO}/preprocess/synthetic_prices.py'), 'Commit preprocess/synthetic_prices.py to GitHub first'
from preprocess.synthetic_prices import synthetic_price

PROJECT_DIR = '/content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk/data/raw'
df = pd.read_parquet(f'{PROJECT_DIR}/data/listings_en_raw.parquet')
print(df.shape)
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/data/listings_en_raw.parquet'

In [ ]:
df['price'] = [synthetic_price(i, t) for i, t in zip(df['item_id'], df['ptype'])]
df['price_currency'] = 'USD'
df['price_is_synthetic'] = True
df[['item_id', 'ptype', 'price', 'price_is_synthetic']].head(10)

In [ ]:
print(df['price'].describe())

med = df.groupby('ptype')['price'].median().sort_values()
print('Cheapest types by median price:')
print(med.head(10))
print('Most expensive types by median price:')
print(med.tail(10))

under50 = df[df['price'] < 50]
print(len(under50), 'items under $50')
print(under50['ptype'].value_counts().head(10))

# Determinism check: running the generator again must give identical prices
again = [synthetic_price(i, t) for i, t in zip(df['item_id'].head(1000), df['ptype'].head(1000))]
assert again == df['price'].head(1000).tolist(), 'Prices are not deterministic'
print('Determinism check passed')

In [ ]:
out = f'{PROJECT_DIR}/data/listings_en_priced.parquet'
df.to_parquet(out)
print('Saved', out, df.shape)